# Exploratory data analysis

*A signal-processing and time-series perspective*

## Can we predict future demand?

We want to forecast **six days ahead, every 30 minutes**. Before choosing a model, we ask: **what historical patterns could support prediction?**

We examine two types of patterns:

- **Temporal patterns:** how demand changes and repeats over time.
- **Driver-related patterns:** how changes in signals such as temperature and humidity correspond to changes in demand. For predictive purposes, we examine correlation rather than causation.

We start with temporal patterns.

In [1]:
# Local kernel: project .venv (Python 3.12); Run All uses the saved AEMO_VIC1 login.
# Spark reads remotely; pandas prepares data, Plotly draws charts, statsmodels decomposes demand.
%pip install -q plotly==7.1.0 pandas==2.3.3 numpy==1.26.4 scipy==1.16.3 statsmodels==0.14.6

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Use final Silver actual demand, not overlapping candidate records.
# OPERATIONAL_DEMAND_LESS_SNSG excludes significant non-scheduled generation; MW measures power.
# Preserve source clock labels, without Melbourne daylight-saving conversion.
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from databricks.connect import DatabricksSession

# Shared chart style from docs/notion/DESIGN_SYSTEM.md.
COLORS = {
    "navy": "#17263C", "signal": "#2F80ED", "reference": "#C98500", "reference_text": "#8A5A00",
    "muted": "#667085", "white": "#FFFFFF",
    "neutral": "#F4F6F8", "border": "#D9E0E8",
}
pio.templates["project"] = go.layout.Template(pio.templates["plotly_white"])
pio.templates["project"].layout.update(
    font=dict(family="Arial, sans-serif", size=14, color=COLORS["navy"]),
    title=dict(font=dict(size=20, color=COLORS["navy"])),
    paper_bgcolor=COLORS["white"], plot_bgcolor=COLORS["white"],
    colorway=[COLORS["signal"], COLORS["reference"], COLORS["muted"]],
    xaxis=dict(
        gridcolor=COLORS["neutral"], zerolinecolor=COLORS["border"],
        tickfont=dict(color=COLORS["muted"], size=12),
        rangeslider=dict(bgcolor=COLORS["neutral"], bordercolor=COLORS["border"], borderwidth=1),
    ),
    yaxis=dict(gridcolor=COLORS["neutral"], zerolinecolor=COLORS["border"], tickfont=dict(color=COLORS["muted"], size=12)),
    annotationdefaults=dict(font=dict(size=12), bgcolor=COLORS["white"], borderpad=3),
)
pio.templates.default = "project"

spark = DatabricksSession.builder.getOrCreate()

DEMAND_TABLE = "workspace.default.demand_vic_30min"
WINDOW_START = None  # Set "2026-09-01" to inspect six days from another date.
HORIZON_DAYS = 6

demand = spark.table(DEMAND_TABLE).selectExpr("CAST(time AS STRING) AS time", "demand").toPandas()
demand["time"] = pd.to_datetime(demand["time"])
demand = demand.sort_values("time").set_index("time")
display(demand.head())

,demand
time,
2025-01-31 09:30:00,3863.0
2025-08-01 04:30:00,5181.0
2025-08-01 05:00:00,5273.0
2025-08-01 05:30:00,5527.0
2025-08-01 06:00:00,5880.0


In [3]:
# Check coverage before interpretation; leave missing half-hours blank rather than interpolate.
assert demand.index.is_unique, "Resolve duplicate timestamps in Silver before plotting."
assert (demand.index == demand.index.floor("30min")).all(), "Expected half-hour interval labels."

expected_times = pd.date_range(demand.index.min(), demand.index.max(), freq="30min")
history = demand.reindex(expected_times)
history.index.name = "time"

display(pd.DataFrame([{
    "first_time": demand.index.min(),
    "last_time": demand.index.max(),
    "stored_rows": len(demand),
    "expected_half_hours": len(history),
    "missing_intervals": len(history) - len(demand),
    "null_demand_values": int(demand["demand"].isna().sum()),
}]))

spacing = demand.index.to_series().diff()
gap_ends = spacing[spacing > pd.Timedelta(minutes=30)]
display(pd.DataFrame({
    "before_gap": gap_ends.index - gap_ends.values,
    "after_gap": gap_ends.index,
    "missing_half_hours": (gap_ends / pd.Timedelta(minutes=30) - 1).astype(int).values,
}))

,first_time,last_time,stored_rows,expected_half_hours,missing_intervals,null_demand_values
0,2025-01-31 09:30:00,2026-10-02 18:00:00,20525,29250,8725,0


,before_gap,after_gap,missing_half_hours
0,2025-01-31 09:30:00,2025-08-01 04:30:00,8725


## 1. Temporal patterns

### Patterns across the history

The slow rise and fall suggests a possible yearly pattern. The main continuous record covers about **14 months, from August 2025 to October 2026**: enough to see its shape, but not to confirm that it repeats each year.

In [4]:
overview = go.Figure(go.Scatter(
    x=history.index, y=history["demand"],
    mode="lines", name="Actual demand", connectgaps=False,
    line=dict(color=COLORS["signal"], width=1),
    hovertemplate="%{x|%a %d %b %Y %H:%M}<br>%{y:,.0f} MW<extra></extra>",
))
overview.update_layout(
    title="Demand history", height=480,
    yaxis_title="Demand (MW)", xaxis_title="Date",
    hovermode="x unified", margin=dict(t=85, b=45),
    xaxis=dict(
        rangeslider=dict(visible=True, thickness=0.12),
        rangeselector=dict(buttons=[
            dict(count=6, label="6 days", step="day", stepmode="backward"),
            dict(count=7, label="1 week", step="day", stepmode="backward"),
            dict(count=1, label="1 month", step="month", stepmode="backward"),
            dict(count=1, label="1 year", step="year", stepmode="backward"),
            dict(label="All", step="all"),
        ]),
    ),
)
overview.show()

### A closer look at daily and weekly repetition

Within six days, peaks and dips appear at similar times each day, suggesting a **daily pattern**. Move the window across the history to compare days and look for differences that might repeat each week.

In [5]:
window_end = demand.index.max().normalize()
window_start = (
    pd.Timestamp(WINDOW_START) if WINDOW_START is not None
    else window_end - pd.Timedelta(days=HORIZON_DAYS)
)
window_end = window_start + pd.Timedelta(days=HORIZON_DAYS)
window_times = pd.date_range(window_start, window_end, freq="30min", inclusive="left")
window = history.reindex(window_times)

display(pd.DataFrame([{
    "window_start_inclusive": window_start,
    "window_end_exclusive": window_end,
    "expected_intervals": len(window),
    "available_values": int(window["demand"].notna().sum()),
}]))

close_up = go.Figure(go.Scatter(
    x=history.index, y=history["demand"],
    mode="lines", name="Actual demand", connectgaps=False,
    line=dict(color=COLORS["signal"], width=2),
    hovertemplate="%{x|%a %d %b %H:%M}<br>%{y:,.0f} MW<extra></extra>",
))
close_up.update_layout(
    title="Six-day view", height=490,
    yaxis_title="Demand (MW)", xaxis_title="Date",
    hovermode="x unified", dragmode="pan",
    xaxis=dict(
        range=[window_start, window_end], dtick=86400000, tickformat="%a\n%d %b",
        rangeslider=dict(visible=True, thickness=0.12),
    ),
    margin=dict(t=65, b=55),
)
close_up.show()

,window_start_inclusive,window_end_exclusive,expected_intervals,available_values
0,2026-09-26,2026-10-02,288,288


### Decomposing cyclical components and the remainder

We start with the hypothesis that demand contains **yearly, weekly and daily cycles**. Decomposition separates these candidate patterns from the **residual**: everything they leave unexplained, including noise, other influences and patterns we may have missed.

Daily and weekly periods are assumptions supplied to the method, so their appearance here is not independent proof. The yearly panel shows slow movement compared with a yearly reference.

Blue shows each component; dashed amber shows a fitted sine wave at its reference period. Together, the components and residual reconstruct demand.

**Variance explained (%)** here means 100 × Var(component) / Var(full fitted signal). It describes component size, not variance explained by a predictive model. Components can overlap or cancel, so these ratios need not add to 100%. Reference sine waves are not counted as components.

In [6]:
# MSTL specifies daily/weekly periods; it does not discover them independently.
# Two-sided historical fitting is descriptive, not a forecast-origin-safe feature.
# Reference sine waves use least squares, not inverse FFT reconstruction.
# MSTL: https://www.statsmodels.org/v0.14.6/generated/statsmodels.tsa.seasonal.MSTL.html
from statsmodels.tsa.seasonal import MSTL
from plotly.subplots import make_subplots
import numpy as np

# Keep the longest uninterrupted run; never bridge missing demand.
observed = history["demand"].dropna()
run_id = observed.index.to_series().diff().ne(pd.Timedelta(minutes=30)).cumsum()
stl_input = observed[run_id.eq(run_id.value_counts().idxmax())].asfreq("30min")
daily_period = 48
weekly_period = 7 * daily_period
trend_window = 14 * daily_period + 1  # Approximately two weeks; the window must be odd.
seasonal_windows = (31, 15)  # Daily cycles and weekly cycles.
assert len(stl_input) > 2 * weekly_period, "MSTL needs more than two complete weeks here."

display(pd.DataFrame([{
    "first_time": stl_input.index.min(),
    "last_time": stl_input.index.max(),
    "half_hours_used": len(stl_input),
}]))

mstl = MSTL(
    stl_input, periods=(daily_period, weekly_period), windows=seasonal_windows,
    stl_kwargs=dict(trend=trend_window, robust=True),
).fit()
components = pd.DataFrame({
    "Yearly pattern": mstl.trend,  # Slow trend, not a confirmed annual cycle.
    "Weekly pattern": mstl.seasonal["seasonal_336"],
    "Daily pattern": mstl.seasonal["seasonal_48"],
    "Residual": mstl.resid,
})

# Component variance relative to the full fitted signal; ratios need not sum to 100%.
component_variance_pct = 100 * components.var(ddof=0) / stl_input.var(ddof=0)

# Fit phase, amplitude and baseline at the three reference periods.
reference_periods = {"Yearly pattern": 365.25, "Weekly pattern": 7.0, "Daily pattern": 1.0}
elapsed_days = np.asarray((components.index - components.index[0]) / pd.Timedelta(days=1))
sine_overlays = pd.DataFrame(index=components.index)
for column, period_days in reference_periods.items():
    angle = 2 * np.pi * elapsed_days / period_days
    basis = np.column_stack([np.ones(len(angle)), np.cos(angle), np.sin(angle)])
    coefficients = np.linalg.lstsq(basis, components[column].to_numpy(), rcond=None)[0]
    sine_overlays[column] = basis @ coefficients

weekly_start = window_end - pd.Timedelta(days=28)
view_ranges = [
    [stl_input.index.min(), stl_input.index.max()], [weekly_start, window_end],
    [window_start, window_end], [weekly_start, window_end],
]
decomposition = make_subplots(
    rows=4, cols=1, shared_xaxes=False, vertical_spacing=0.11,
    subplot_titles=components.columns.tolist(),
)
for row, column in enumerate(components.columns, start=1):
    decomposition.add_annotation(
        x=0.99, y=1.025, xref="x domain", yref="y domain",
        text=f"Variance explained: {component_variance_pct[column]:.1f}%", showarrow=False,
        xanchor="right", yanchor="bottom", bgcolor=COLORS["white"], row=row, col=1,
    )
    decomposition.add_trace(go.Scatter(
        x=components.index, y=components[column], mode="lines", name=column,
        line=dict(color=COLORS["muted"] if column == "Residual" else COLORS["signal"], width=1.8),
        hovertemplate="%{x|%a %d %b %Y %H:%M}<br>%{y:,.0f} MW<extra>%{fullData.name}</extra>",
    ), row=row, col=1)
    decomposition.update_yaxes(
        title_text="MW", gridcolor=COLORS["neutral"], zerolinecolor=COLORS["border"],
        tickfont=dict(color=COLORS["muted"], size=12), row=row, col=1,
    )
    decomposition.update_xaxes(
        range=view_ranges[row - 1], title_text="Date · 30-minute data",
        tickformat="%b %Y" if row == 1 else "%d %b<br>%H:%M" if row == 3 else "%d %b %Y",
        dtick="M2" if row == 1 else 86400000 if row == 3 else 7 * 86400000,
        gridcolor=COLORS["neutral"], tickfont=dict(color=COLORS["muted"], size=12),
        rangeslider=dict(visible=True, thickness=0.02, bgcolor=COLORS["neutral"], bordercolor=COLORS["border"], borderwidth=1),
        row=row, col=1,
    )
    if column in reference_periods:
        decomposition.add_trace(go.Scatter(
            x=sine_overlays.index, y=sine_overlays[column], mode="lines",
            name=f"{reference_periods[column]:g}-day sine reference",
            line=dict(color=COLORS["reference"], width=2.2, dash="dash"),
            hovertemplate="%{x|%a %d %b %Y %H:%M}<br>%{y:,.0f} MW<extra>%{fullData.name}</extra>",
        ), row=row, col=1)
        # Place each arrow on a reference point inside its initial view.
        point = sine_overlays[column].idxmax() if row == 1 else sine_overlays.loc[view_ranges[row - 1][0]:window_end, column].idxmax()
        decomposition.add_annotation(
            x=point, y=sine_overlays.loc[point, column],
            text=f"{reference_periods[column]:g}-day sine reference", showarrow=True,
            arrowhead=2, ax=-80, ay=40 if row == 1 else -40,
            arrowcolor=COLORS["reference"], font=dict(color=COLORS["reference_text"]), row=row, col=1,
        )
decomposition.update_layout(
    title="Demand components", height=1520, showlegend=False,
    hovermode="x unified", dragmode="pan", margin=dict(t=85, b=60),
)
decomposition.show()

,first_time,last_time,half_hours_used
0,2025-08-01 04:30:00,2026-10-02 18:00:00,20524


### Quantifying cyclical strength with the Fourier transform

The **Fourier transform** quantifies the strength of oscillations in the signal. Peaks show the periods with the largest contributions; the axis uses days so we can recognise daily, weekly and slower movement.

The **fundamental** is the base cycle. **Harmonics** repeat 2, 3, 4, … times faster and combine with it to turn a pure sine wave into the shape we observe. For a daily cycle, these include 12, 8 and 6 hours. A harmonic can be stronger than the fundamental.

In [7]:
# Centre and Hann-taper each component; magnitude is amplitude-scaled in MW, not energy.
# The 365.25-day reference is not an exact FFT bin in this roughly 14-month record.
# Harmonic periods are P/n; not every shorter-period peak is a harmonic.
# Fourier terms: https://otexts.com/fpp3/useful-predictors.html
signals = components[["Yearly pattern", "Weekly pattern", "Daily pattern"]]
sample_days = 1 / 48
sample_count = len(signals)
taper = np.hanning(sample_count)
centered = signals.to_numpy() - signals.mean().to_numpy()
transform = np.fft.rfft(centered * taper[:, None], axis=0)
frequencies = np.fft.rfftfreq(sample_count, d=sample_days)

# One-sided amplitude scaling; zero frequency has no finite period.
magnitudes = 2 * np.abs(transform) / taper.sum()
if sample_count % 2 == 0:
    magnitudes[-1] /= 2
spectrum = pd.DataFrame(
    magnitudes[1:], index=1 / frequencies[1:], columns=signals.columns,
).sort_index()
spectrum.index.name = "period_days"
magnitude_limit = spectrum.to_numpy().max() * 1.2

fourier_plot = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.15,
    subplot_titles=signals.columns.tolist(),
)
for row, column in enumerate(signals.columns, start=1):
    fourier_plot.add_trace(go.Scatter(
        x=spectrum.index, y=spectrum[column], mode="lines", name=column,
        line=dict(color=COLORS["signal"], width=2),
        hovertemplate="Period: %{x:.3f} days<br>Magnitude: %{y:,.1f} MW<extra>%{fullData.name}</extra>",
    ), row=row, col=1)
    reference = reference_periods[column]
    # Highlight the fundamental and listed harmonics; shading does not filter the spectrum.
    fourier_plot.add_vrect(
        x0=reference * 0.95 if row == 1 else reference / 4 * 0.95,
        x1=reference * 1.05, fillcolor=COLORS["signal"],
        opacity=0.06, line_width=0, layer="below", row=row, col=1,
    )
    fourier_plot.add_vline(x=reference, line_dash="dot", line_color=COLORS["reference"], row=row, col=1)
    band = spectrum.loc[spectrum.index >= 30, column] if row == 1 else spectrum.loc[(spectrum.index > reference * 0.95) & (spectrum.index < reference * 1.05), column]
    peak_period = band.idxmax()
    label = "Long-period peak" if row == 1 else f"Fundamental<br>~{reference:g} day" + ("s" if reference > 1 else "")
    fourier_plot.add_annotation(
        x=np.log10(peak_period), y=band.loc[peak_period], text=label,
        showarrow=True, arrowhead=2, ax=-100 if row == 1 else 65, ay=-40,
        arrowcolor=COLORS["signal"] if row == 1 else COLORS["reference"],
        font=dict(color=COLORS["signal"] if row == 1 else COLORS["reference_text"]), row=row, col=1,
    )
    if row == 1:
        fourier_plot.add_annotation(
            x=np.log10(reference), y=0, text="365.25-day reference", showarrow=False,
            yshift=12, xanchor="right", font=dict(color=COLORS["reference_text"]), row=row, col=1,
        )
    else:
        # Harmonics have periods P/n, not arbitrary periods below P.
        for harmonic in [2, 3, 4]:
            fourier_plot.add_vline(
                x=reference / harmonic, line_color=COLORS["signal"],
                line_width=1, line_dash="dot", opacity=0.3, layer="below", row=row, col=1,
            )
        fourier_plot.add_annotation(
            x=np.log10(reference / 3), y=magnitude_limit * 0.86,
            text="Fundamental + harmonics<br>P, P/2, P/3, P/4", showarrow=False,
            font=dict(color=COLORS["signal"]), row=row, col=1,
        )
    fourier_plot.update_xaxes(
        type="log", title_text="Period (days)", showticklabels=True,
        tickvals=[0.25, 0.5, 1, 3.5, 7, 30, 365.25],
        ticktext=["0.25", "0.5", "1", "3.5", "7", "30", "365.25"],
        range=[np.log10(spectrum.index.min()), np.log10(spectrum.index.max())],
        gridcolor=COLORS["neutral"], tickfont=dict(color=COLORS["muted"], size=12), row=row, col=1,
    )
    fourier_plot.update_yaxes(
        title_text="Magnitude (MW)", range=[0, magnitude_limit],
        gridcolor=COLORS["neutral"], zerolinecolor=COLORS["border"], row=row, col=1,
    )
half_day_band = spectrum.loc[(spectrum.index > 0.49) & (spectrum.index < 0.51), "Daily pattern"]
half_day_peak = half_day_band.idxmax()
fourier_plot.add_annotation(
    x=np.log10(half_day_peak), y=half_day_band.loc[half_day_peak], text="2nd harmonic<br>0.5 day (12 h)",
    showarrow=True, arrowhead=2, ax=-85, ay=30,
    arrowcolor=COLORS["signal"], font=dict(color=COLORS["signal"]), row=3, col=1,
)
fourier_plot.update_layout(
    title="Fourier spectrum", height=1000, showlegend=False,
    margin=dict(t=85, b=60),
)
fourier_plot.show()

### So, can we project temporal patterns into future demand? / Conclusion of the analysis

Yes: daily, weekly and slower yearly-scale temporal patterns give us **historical structure that we can project into the future**.

## 2. Driver-related patterns

We compare a single weather component with demand, using temperature and humidity from the nine selected locations.

### Standardizing weather and retaining one component

Use temperature and humidity from all nine locations: **18 signals**, each centred to mean zero and scaled to unit variance. PCA combines them; the table shows how much weather variance each component explains. We retain **PC1 only**, highlighted in the first row. The remaining components are shown for reference only.

Weather is hourly. For comparison, demand is averaged from the two half-hours ending at each hour. Only the continuous demand history and matching complete weather hours are used. From each daily weather forecast, we keep the first 24 hours to avoid overlapping forecasts for the same time.

In [8]:
WEATHER_TABLE = "workspace.default.weather_forecast_vic_hourly"

# Use only the continuous demand period already analysed above.
weather = (
    spark.table(WEATHER_TABLE)
    .where(f"time >= '{stl_input.index.min()}' AND time <= '{stl_input.index.max()}'")
    .where("time > forecast_origin AND time <= forecast_origin + INTERVAL 24 HOURS")
    .selectExpr("CAST(time AS STRING) AS time", "location", "temperature", "humidity")
    .toPandas()
)
weather["time"] = pd.to_datetime(weather["time"])
weather_wide = weather.pivot(index="time", columns="location", values=["temperature", "humidity"])
weather_wide.columns = [
    f"{variable}_{location.lower().replace(' ', '_')}"
    for variable, location in weather_wide.columns
]
weather_columns = weather_wide.columns.tolist()
assert len(weather_columns) == 18, "Expected temperature and humidity at nine locations."

# Demand timestamps label interval ends: hour t averages demand at t-30 min and t.
hourly_demand = stl_input.resample("h", closed="right", label="right").agg(["mean", "count"])
hourly_demand = hourly_demand.loc[hourly_demand["count"].eq(2), "mean"].rename("demand")
matched = weather_wide.join(hourly_demand, how="inner").dropna()
assert not matched.empty and matched.index.is_unique
assert matched[weather_columns].std(ddof=0).gt(0).all()

# Standardize weather only; demand is not an input to PCA.
weather_scaled = (matched[weather_columns] - matched[weather_columns].mean()) / matched[weather_columns].std(ddof=0)

# SVD computes PCA using NumPy, without another dependency.
_, singular_values, directions = np.linalg.svd(weather_scaled.to_numpy(), full_matrices=False)
component_variance = singular_values**2 / (len(weather_scaled) - 1)
variance_share = component_variance / component_variance.sum()
pca_variance = pd.DataFrame({
    "component": [f"PC{i + 1}" for i in range(len(variance_share))],
    "variance": component_variance,
    "explained_variance_pct": 100 * variance_share,
    "cumulative_pct": 100 * variance_share.cumsum(),
})

print(f"Matched hourly history: {matched.index.min()} to {matched.index.max()} | {len(matched):,} hours | 18 weather signals")
print(f"PC1 retains {100 * variance_share[0]:.1f}% of standardized weather variance.")
from IPython.display import HTML
pc1_style = "<style>#pca-variance tbody tr:first-child {background: #EAF3FF; color: #17263C; font-weight: bold;}</style>"
display(HTML(pc1_style + pca_variance.round(2).to_html(index=False, table_id="pca-variance")))


Matched hourly history: 2025-08-01 05:00:00 to 2026-10-02 18:00:00 | 10,118 hours | 18 weather signals
PC1 retains 82.5% of standardized weather variance.


component,variance,explained_variance_pct,cumulative_pct
PC1,14.86,82.52,82.52
PC2,2.41,13.38,95.90
PC3,0.23,1.26,97.16
PC4,0.20,1.10,98.26
PC5,0.15,0.84,99.10
PC6,0.05,0.28,99.38
PC7,0.04,0.20,99.58
PC8,0.03,0.14,99.72
PC9,0.02,0.10,99.82
PC10,0.01,0.07,99.89


### Weather PC1 alongside demand

Both curves are scaled to **mean zero and unit variance** over the same matched hours. Values show standard deviations from their own means, not MW or temperature. The top-right label reports Pearson correlation over all matched hours.

PC1 captures weather variation, not demand-predictive importance. For this overlay only, PC1 is sign-reversed for display; the correlation uses those displayed values. The component panels retain their documented display directions. Scaling and PCA are fitted on this historical period for exploration, not forecast evaluation.

In [9]:
weather_pc1 = pd.Series(weather_scaled.to_numpy() @ directions[0], index=matched.index, name="Weather PC1")
comparison_signals = pd.concat([matched["demand"].rename("Demand"), weather_pc1], axis=1)
comparison_scaled = (comparison_signals - comparison_signals.mean()) / comparison_signals.std(ddof=0)

# Preserve missing hours as gaps instead of joining across them.
comparison_scaled = comparison_scaled.reindex(
    pd.date_range(comparison_scaled.index.min(), comparison_scaled.index.max(), freq="h")
)
# Reverse only this overlay; keep the underlying PCA and decomposition unchanged.
comparison_display = comparison_scaled.copy()
comparison_display["Weather PC1"] *= -1
weather_demand_plot = go.Figure()
for column, color in [("Demand", COLORS["signal"]), ("Weather PC1", COLORS["reference"])]:
    weather_demand_plot.add_trace(go.Scatter(
        x=comparison_display.index, y=comparison_display[column], mode="lines", name=column,
        line=dict(color=color, width=2), connectgaps=False,
        hovertemplate="%{x|%a %d %b %Y %H:%M}<br>%{y:.2f} standard deviations<extra>%{fullData.name}</extra>",
    ))
weather_demand_plot.update_layout(
    title="Weather PC1 and demand", height=540, hovermode="x unified",
    yaxis_title="Standard deviations from the mean", xaxis_title="Date · hourly comparison",
    legend=dict(orientation="h", y=1.10, x=1, xanchor="right"), margin=dict(t=100, b=50),
    xaxis=dict(
        rangeslider=dict(visible=True, thickness=0.12),
        rangeselector=dict(buttons=[
            dict(count=6, label="6 days", step="day", stepmode="backward"),
            dict(count=1, label="1 month", step="month", stepmode="backward"),
            dict(label="All", step="all"),
        ]),
    ),
)
weather_demand_plot.add_annotation(
    x=0.99, y=0.98, xref="paper", yref="paper",
    text=f"Pearson r (display) = {comparison_display['Demand'].corr(comparison_display['Weather PC1']):+.3f}",
    showarrow=False, xanchor="right", yanchor="top", bgcolor=COLORS["white"],
)
weather_demand_plot.show()


### Correlation with demand

Pearson correlation measures same-hour linear association: from −1 (opposite movement) to +1 (same-direction movement), with zero indicating no linear association. PC1's sign is arbitrary; this is a historical comparison, not a forecasting score.

In [10]:
correlation_pairs = comparison_display[["Weather PC1", "Demand"]].dropna()
pc1_correlation = correlation_pairs["Weather PC1"].corr(correlation_pairs["Demand"])
print(f"Pearson correlation (displayed PC1 vs demand): {pc1_correlation:.3f} | {len(correlation_pairs):,} matched hours")

Pearson correlation (displayed PC1 vs demand): 0.361 | 10,118 matched hours


### Comparing the temporal components

Signals are standardized over the same longest uninterrupted overlap before decomposition; components retain their relative amplitudes. Labels report full-history correlations and **variance explained (%)**: component variance divided by its own full signal variance. These percentages need not sum to 100% because components can overlap or cancel. The yearly panel shows slow trend, not a confirmed annual cycle. For display only, PC1’s yearly, weekly and daily components have their signs reversed; the residual is unchanged. These individually oriented curves do not reconstruct one PC1 signal.


Blue shading behind the residual curves marks larger **demand residual energy**: the trailing 24-hour mean of squared residuals. Darker means larger swings, whether positive or negative. Square-root colour spacing makes quieter periods visible without clipping extremes; the scale stays fixed across dates. Hover shows the unchanged value in squared standard deviations, not electricity consumption.


In [11]:
# Use the same continuous hourly run for both signals; never bridge missing hours.
complete_signals = comparison_scaled[["Demand", "Weather PC1"]].dropna()
comparison_runs = complete_signals.index.to_series().diff().ne(pd.Timedelta(hours=1)).cumsum()
component_input = complete_signals.loc[comparison_runs.eq(comparison_runs.value_counts().idxmax())].asfreq("h")
component_input = (component_input - component_input.mean()) / component_input.std(ddof=0)
assert component_input.notna().all().all() and len(component_input) > 2 * 168

# Same smoothing durations as before, expressed in hourly rather than half-hourly samples.
paired_components = {}
for signal in component_input.columns:
    fitted = MSTL(
        component_input[signal], periods=(24, 168), windows=(31, 15),
        stl_kwargs=dict(trend=337, robust=True),
    ).fit()
    paired_components[signal] = pd.DataFrame({
        "Yearly pattern": fitted.trend,
        "Weekly pattern": fitted.seasonal["seasonal_168"],
        "Daily pattern": fitted.seasonal["seasonal_24"],
        "Residual": fitted.resid,
    })
    assert np.allclose(paired_components[signal].sum(axis=1), component_input[signal])

paired_variance_pct = pd.DataFrame({
    signal: 100 * parts.var(ddof=0) / component_input[signal].var(ddof=0)
    for signal, parts in paired_components.items()
})

component_end = component_input.index.max()
component_ranges = [
    [component_input.index.min(), component_end],
    [component_end - pd.Timedelta(days=28), component_end],
    [component_end - pd.Timedelta(days=6), component_end],
    [component_end - pd.Timedelta(days=28), component_end],
]
paired_decomposition = make_subplots(
    rows=4, cols=1, shared_xaxes=False, vertical_spacing=0.11,
    subplot_titles=paired_components["Demand"].columns.tolist(),
)
for row, component in enumerate(paired_components["Demand"].columns, start=1):
    pc1_display_direction = 1 if component == "Residual" else -1
    panel_signals = [("Demand", COLORS["signal"]), ("Weather PC1", COLORS["reference"])]
    if component == "Residual":
        # Squaring counts both upward and downward excursions; smooth over one day.
        residual_values = np.concatenate([parts["Residual"].to_numpy() for parts in paired_components.values()])
        residual_padding = 0.05 * np.ptp(residual_values)
        residual_low, residual_high = residual_values.min() - residual_padding, residual_values.max() + residual_padding
        residual_energy = paired_components["Demand"]["Residual"].pow(2).rolling(24, min_periods=24).mean()
        paired_decomposition.add_trace(go.Heatmap(
            x=component_input.index, y=[residual_low, residual_high],
            z=[residual_energy.to_numpy(), residual_energy.to_numpy()],
            # Square-root colour spacing reveals quieter periods without clipping extremes.
            colorscale=[[0, "#FFFFFF"], [0.0025, "#F4F7FC"], [0.0225, "#E3EDF9"],
                        [0.09, "#C9DDF5"], [0.25, "#B3CFF1"], [1, "#8EB5E3"]],
            zmin=0, zmax=residual_energy.max(), showscale=True,
            colorbar=dict(
                orientation="h", x=0.99, xanchor="right",
                y=paired_decomposition.layout.yaxis4.domain[1] - 0.005, yanchor="top",
                len=0.23, thickness=8, bgcolor=COLORS["white"],
                title=dict(text="Demand residual energy · 24h", side="top", font=dict(size=11)),
                tickvals=[0, residual_energy.max()], ticktext=["Low", "High"],
                tickfont=dict(size=11), outlinecolor=COLORS["border"], outlinewidth=1,
            ),
            name="Demand residual energy",
            hovertemplate="%{x|%d %b %Y %H:%M}<br>24-hour demand residual energy: %{z:.2f} SD²<extra></extra>",
        ), row=row, col=1)
        paired_decomposition.update_yaxes(range=[residual_low, residual_high], row=row, col=1)
    for signal, color in panel_signals:
        paired_decomposition.add_trace(go.Scatter(
            x=component_input.index,
            y=paired_components[signal][component] * (pc1_display_direction if signal == "Weather PC1" else 1),
            mode="lines", name="Weather PC1 (display)" if signal == "Weather PC1" else signal,
            legendgroup=signal, showlegend=row == 1,
            line=dict(color=color, width=1.8), connectgaps=False,
            hovertemplate="%{x|%a %d %b %Y %H:%M}<br>%{y:.2f} standard deviations<extra>%{fullData.name}</extra>",
        ), row=row, col=1)
    component_correlation = paired_components["Demand"][component].corr(pc1_display_direction * paired_components["Weather PC1"][component])
    correlation_label = f"Pearson r (display): Demand–PC1 {component_correlation:+.3f}"
    variance_label = f"Variance explained: Demand {paired_variance_pct.loc[component, 'Demand']:.1f}% | PC1 {paired_variance_pct.loc[component, 'Weather PC1']:.1f}%"
    paired_decomposition.add_annotation(
        x=0.99, y=1.025, xref="x domain", yref="y domain",
        text=f"{correlation_label}<br>{variance_label}",
        showarrow=False, xanchor="right", yanchor="bottom", bgcolor=COLORS["white"], row=row, col=1,
    )
    paired_decomposition.update_yaxes(title_text="Standard deviations", row=row, col=1)
    paired_decomposition.update_xaxes(
        range=component_ranges[row - 1], title_text="Date · hourly data",
        tickformat="%b %Y" if row == 1 else "%d %b<br>%H:%M" if row == 3 else "%d %b %Y",
        dtick="M2" if row == 1 else 86400000 if row == 3 else 7 * 86400000,
        rangeslider=dict(visible=True, thickness=0.02),
        row=row, col=1,
    )
paired_decomposition.update_layout(
    title="Demand and weather components", height=1520, hovermode="x unified", dragmode="pan",
    legend=dict(orientation="h", x=1, xanchor="right", y=1.04),
    margin=dict(t=110, b=60),
)
paired_decomposition.show()


### Correlation during high-energy periods

*10 August 2025–11 June 2026. High-energy groups overlap.*


In [12]:
residual_pairs = pd.DataFrame({
    "Demand": paired_components["Demand"]["Residual"],
    "Weather PC1": paired_components["Weather PC1"]["Residual"],
})
# Thresholds use the full history; the first 23 hours lack a complete energy window.
energy_pairs = residual_pairs.assign(energy=residual_energy).dropna()
energy_rows = [{
    "Demand residual energy": "Full period (reference)",
    "Hours": len(residual_pairs),
    "Pearson r": residual_pairs["Demand"].corr(residual_pairs["Weather PC1"]),
}]
for label, quantile, high in [
    ("Lower 90%", 0.90, False), ("Highest 10%", 0.90, True),
    ("Highest 5%", 0.95, True),
]:
    threshold = energy_pairs["energy"].quantile(quantile)
    selected = energy_pairs.loc[energy_pairs["energy"].ge(threshold) if high else energy_pairs["energy"].lt(threshold)]
    energy_rows.append({
        "Demand residual energy": label, "Hours": len(selected),
        "Pearson r": selected["Demand"].corr(selected["Weather PC1"]),
    })
display(pd.DataFrame(energy_rows).round({"Pearson r": 3}))


,Demand residual energy,Hours,Pearson r
0,Full period (reference),7320,0.184
1,Lower 90%,6567,-0.040
2,Highest 10%,730,0.489
3,Highest 5%,365,0.569


### Are there driver-related patterns in demand? / Conclusion of the analysis

Both signals share daily and slower yearly-scale cycles, which may help a model recognize demand’s cyclical behaviour.

To examine driver-related patterns beyond these shared cycles, we also compare their residuals.

**Yes: weather PC1 and demand tend to move together during high-energy demand periods**, supporting driver-related patterns. Other possible drivers were not examined.


### Pattern summary

| Type | Pattern summary |
| --- | --- |
| **Time** | • **Yearly:** seasonal cycle of approximately **365.25 days**.<br>• **Daily:** fundamental **1 day**; harmonics **0.5, 0.333 and 0.25 days**.<br>• **Weekly:** fundamental **7 days**; harmonics approximately **3.5, 2.33 and 1.75 days**.<br>• **Noise assumption:** components outside the retained signal band are treated as noise. |
| **Driver** | • Weather PC1 and demand share daily and yearly patterns.<br>• High residual correlation during high-energy demand periods. |